# ⚡ MedGemma-4B API Server
> **Runtime → Change runtime type → T4 GPU** قبل ما تشغّل


In [ ]:
# ── Cell 1: تثبيت المكتبات ──────────────────────────────────────
!pip install -q transformers accelerate fastapi uvicorn pyngrok pillow torch torchvision
print('✅ Done!')

In [ ]:
# ── Cell 2: تحميل الموديل ───────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'google/medgemma-4b-it'

print(f'🔄 Loading {MODEL_ID} ...')
print(f'🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU!"}')

from huggingface_hub import login
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print('✅ HF Login OK')
except:
    print('⚠️  ضيف HF_TOKEN في Colab Secrets')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
    token=True,
)
model.eval()
print('✅ MedGemma-4B loaded!')

In [ ]:
# ── Cell 3: FastAPI Server ───────────────────────────────────────
import threading, base64, io, time
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List, Any
from PIL import Image
import torch

app = FastAPI(title='MedGemma OpenAI-Compatible API')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*']
)


# ── Schemas ──────────────────────────────────────────────────────
class Message(BaseModel):
    role: str
    content: Any   # Any بدل Union — بيقبل str أو list بدون مشكلة

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = 'google/medgemma-4b-it'
    messages: List[Message]
    max_tokens: Optional[int] = 1024
    temperature: Optional[float] = 0.0


# ── normalize_messages ───────────────────────────────────────────
def normalize_messages(messages):
    """
    بيتعامل مع كل أشكال الـ messages:
      - content: 'string'
      - content: [{'type':'text','text':'...'}]
      - content: [{'type':'image_url','image_url':{'url':'data:...'}}]
      - system + user messages مع بعض
    """
    text_parts = []
    images     = []

    for msg in messages:
        content = msg.content

        # حالة 1: نص مباشر
        if isinstance(content, str):
            if msg.role == 'system':
                text_parts.insert(0, f'[SYSTEM]: {content}')
            else:
                text_parts.append(content)

        # حالة 2: list of parts
        elif isinstance(content, list):
            for part in content:
                p = part if isinstance(part, dict) else getattr(part, '__dict__', {})
                ptype = p.get('type', '')
                if ptype == 'text':
                    text_parts.append(p.get('text', '') or '')
                elif ptype == 'image_url':
                    url = (p.get('image_url') or {}).get('url', '')
                    if url.startswith('data:image'):
                        b64 = url.split(',', 1)[1]
                        img = Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')
                        images.append(img)

        # حالة 3: dict واحد
        elif isinstance(content, dict):
            if content.get('type') == 'text':
                text_parts.append(content.get('text', ''))

    return '\n'.join(t for t in text_parts if t).strip(), images


# ── run_inference ────────────────────────────────────────────────
def run_inference(text: str, images: list, max_tokens: int = 1024) -> str:
    if images:
        # مع صورة
        inputs = processor(
            text=f'<start_of_turn>user\n{text}<end_of_turn>\n<start_of_turn>model\n',
            images=images[0],
            return_tensors='pt',
        ).to(model.device)
    else:
        # نص فقط — apply_chat_template بـ tokenize=False الأول
        chat        = [{'role': 'user', 'content': text}]
        prompt_text = processor.apply_chat_template(
            chat,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = processor(
            text=prompt_text,
            return_tensors='pt',
        ).to(model.device)

    input_len = inputs['input_ids'].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
        )
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Endpoints ────────────────────────────────────────────────────
@app.post('/v1/chat/completions')
def chat_completions(req: ChatCompletionRequest):
    try:
        text, images = normalize_messages(req.messages)
        if not text and not images:
            raise HTTPException(400, 'No content found in messages')
        result = run_inference(text, images, max_tokens=req.max_tokens or 1024)
        return {
            'id': f'chatcmpl-{int(time.time())}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': req.model or MODEL_ID,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': result},
                'finish_reason': 'stop'
            }],
            'usage': {'prompt_tokens': -1, 'completion_tokens': -1, 'total_tokens': -1}
        }
    except HTTPException:
        raise
    except Exception as e:
        import traceback
        raise HTTPException(500, f'{e}\n{traceback.format_exc()}')

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': MODEL_ID}


# ── تشغيل السيرفر ────────────────────────────────────────────────
thread = threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
)
thread.start()
time.sleep(2)
print('✅ Server running on port 8000!')

# اختبار محلي سريع
import requests, warnings
warnings.filterwarnings('ignore')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={
            'model': 'test',
            'messages': [{'role': 'user', 'content': 'Say OK in one word'}],
            'max_tokens': 10
        },
        timeout=60
    )
    if r.status_code == 200:
        ans = r.json()['choices'][0]['message']['content']
        print(f'🧪 Test OK → {ans}')
    else:
        print(f'🧪 Test FAILED: {r.text[:300]}')
except Exception as e:
    print(f'🧪 Test error: {e}')

In [ ]:
# ── Cell 4: ngrok ────────────────────────────────────────────────
from pyngrok import ngrok
from google.colab import userdata

try:
    ngrok_token = userdata.get('NGROK_TOKEN')
except:
    ngrok_token = '33vuxDvb47LaYZ5smYAMTaTs0y3_243tra4bXTH3JuNhLhZsZ'

ngrok.set_auth_token(ngrok_token)
tunnel     = ngrok.connect(8000)
public_url = tunnel.public_url    # الـ URL الصح بدون كلام زيادة

print('=' * 55)
print(f'🌐  PUBLIC URL:  {public_url}')
print('=' * 55)
print()
print('📋 حط الـ URL ده في PDF Agent — Cell 2:')
print(f'   LOCAL_API_URL = "{public_url}/v1/chat/completions"')
print(f'   LOCAL_API_KEY = "any-key"')
print(f'   MODEL_NAME    = "google/medgemma-4b-it"')